# Lecture 26: Hierarchical and Empirical Bayes — Sharing Strength Across Many Problems

**Data 145, Spring 2026: Evidence and Uncertainty**
**Instructors:** Ani Adhikari, William Fithian

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Bayesian scheme (prior / likelihood / posterior)
COLOR_LIKELIHOOD = 'black'
COLOR_POSTERIOR = 'steelblue'
COLOR_PRIOR = 'steelblue'
MULTI_BLUES = ['#08519c', '#3182bd', '#6baed6', '#9ecae1']

# Comparing-estimators scheme (IBM colorblind-safe)
COLOR_MLE = '#648FFF'        # MLE / no shrinkage
COLOR_EB = '#785EF0'         # empirical Bayes
COLOR_HB = '#DC267F'         # hierarchical Bayes
COLOR_ORACLE = '#000000'     # oracle (known tau^2)

# Three distinct colors for individual trace plots (colorblind-safe)
TEAM_COLORS = ['#648FFF', '#FE6100', '#DC267F']  # blue, orange, magenta

plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

rng = np.random.default_rng(145)

## Introduction

In Lecture 7 we saw the **Normal-Normal conjugate model** for a single mean: observing $X \sim N(\theta, 1)$ with prior $\theta \sim N(\mu_0, \tau^2)$, the posterior mean is a precision-weighted average of the data and the prior. In Lecture 8 we saw **hierarchical Bayes** in action: the Beta-Binomial baseball model, where hundreds of players share a common $\text{Beta}(\alpha, \beta)$ prior that is itself learned from the data. In Worksheet 4, we implemented a Gibbs sampler for an analogous hierarchical **Gamma-Poisson** model of English Premier League goal-scoring rates. And yesterday (Lecture 25) we studied MCMC and the Gibbs sampler more carefully, with burn-in, thinning, and trace plots.

Today we study the **hierarchical Gaussian model** — the cleanest mathematical setting in which to understand *why* sharing strength across many parallel problems actually works.

1. **Setup and a running example.** The Gaussian sequence model $X_i \sim N(\theta_i, 1)$, $i = 1, \ldots, d$. A simulated dataset we come back to throughout the lecture.
2. **Oracle Bayes with $(\mu, \tau^2)$ known.** The Bayes estimator is a **convex combination** of the prior mean $\mu$ and the data $X_i$: $\hat\theta_i = \zeta \mu + (1 - \zeta) X_i$, where $\zeta = 1/(1 + \tau^2)$. Equivalently, a linear function of $X_i$ with intercept $\zeta\mu$ and slope $1 - \zeta$.
3. **The money formula.** When $(\mu, \tau^2)$ are unknown, the tower property gives the hierarchical-Bayes estimator by applying the *same linear function* with the posterior means of the two coefficients: $\hat\theta_i^{\text{HB}} = E[\zeta \mu \mid X] + (1 - E[\zeta \mid X]) X_i$. Each $\theta_i$ is estimated via its own $X_i$, but through coefficients learned from the *full* data set.
4. **Reading $(\mu, \tau^2)$ off the marginal.** Since $X_i$ is marginally $N(\mu, 1 + \tau^2)$, empirical Bayes estimates $\hat\zeta = 1/S^2$ and $\widehat{\zeta\mu} = \bar X / S^2$ from the familiar sample mean and variance.
5. **Concentration demonstrations.** With many parallel problems, the posteriors on the hyperparameters concentrate — even though the posteriors on individual parameters do not. We see this in the Gaussian-Gaussian model (where everything is explicit) and then in the Gamma-Poisson Premier League model via Gibbs trace plots.

**The central insight.** With many parallel problems, the hyperparameters become effectively known, even while the individual parameters remain uncertain. That is the engine behind "borrowing strength," and the reason hierarchical and empirical Bayes work so well in practice.

---

## 1. Recap: Normal-Normal Conjugacy and Hierarchical Bayes

### 1.1 One mean, Normal-Normal (Lecture 7)

Observing $X \sim N(\theta, 1)$ with prior $\theta \sim N(\mu_0, \tau^2)$, the posterior is
$$\theta \mid X \sim N\!\left(\frac{\tau^2}{1+\tau^2} X + \frac{1}{1+\tau^2}\mu_0,\ \frac{\tau^2}{1+\tau^2}\right),$$
a precision-weighted average of the data and the prior.

### 1.2 Hierarchical Bayes: the idea (Lecture 8)

When we have *many* similar problems, we can model them as drawn from a common population distribution, and let the data inform that distribution:

- **Baseball (Lecture 8).** $p_i \stackrel{iid}{\sim} \text{Beta}(\alpha, \beta)$, $X_i \mid p_i \sim \text{Binomial}(n_i, p_i)$. The hyperparameters $(\alpha, \beta)$ are learned from all the players together; each player's posterior mean is shrunk toward the group mean.
- **Premier League (Worksheet 4).** $\theta_i \stackrel{iid}{\sim} \text{Gamma}(\alpha, \beta)$, $X_{ij} \mid \theta_i \sim \text{Poisson}(\theta_i)$. A Gibbs sampler computes the joint posterior of all the team rates $\theta_i$ and the hyperparameter $\beta$.

### 1.3 What's new today

We study the **Gaussian many-means** version, where the whole story unfolds in closed form. The clean math will let us answer the question:

> *Why* does sharing strength across many parallel problems work?

The answer, in one sentence: **with many parallel problems, the hyperparameter becomes well-identified from the data, so we effectively know the optimal amount of shrinkage.**

---

## 2. The Gaussian Sequence Model and a Running Example

### 2.1 The model

Observe $d$ independent observations
$$X_i \sim N(\theta_i, 1), \qquad i = 1, \ldots, d.$$

The parameter of interest is the whole vector $\theta = (\theta_1, \ldots, \theta_d) \in \mathbb R^d$. If we treated these as $d$ unrelated problems, the obvious estimator for each $\theta_i$ is the MLE $\hat\theta_i = X_i$, with per-coordinate MSE $E_{\theta_i}[(X_i - \theta_i)^2] = 1$ and total MSE $d$.

But what if we suspect the $\theta_i$'s are "similar" — drawn from a common population? Can we do better by analyzing them together?

### 2.2 A running example

Throughout the lecture we'll use a single simulated dataset. Take $d = 200$, with true means drawn from $N(\mu, \tau^2)$ for $\mu = 3$ and $\tau^2 = 4$ (so the $\theta_i$'s are centered at $3$ with typical spread $\pm 2$) and $X_i \mid \theta_i \sim N(\theta_i, 1)$. We choose $d = 200$ deliberately large so that the hyperparameters $(\mu, \tau^2)$ are very well identified from the data — that's the key feature we'll exploit. We'll generate the data in §5, once we're ready to use it.

---

## 3. Oracle Bayes: $(\mu, \tau^2)$ Known

### 3.1 The hierarchical Gaussian model

Place a common $N(\mu, \tau^2)$ prior on the $\theta_i$'s:
$$\theta_i \stackrel{iid}{\sim} N(\mu, \tau^2), \qquad X_i \mid \theta_i \sim N(\theta_i, 1).$$

Coordinate by coordinate, this is exactly the Normal-Normal conjugate model from Lecture 7, with prior mean $\mu$ and prior variance $\tau^2$.

### 3.2 Posterior, coordinate by coordinate

From Lecture 7 (prior mean now $\mu$ instead of $0$):
$$\theta_i \mid \mu, \tau^2,\, X \sim N\!\left(\mu + \frac{\tau^2}{1 + \tau^2}(X_i - \mu),\ \frac{\tau^2}{1 + \tau^2}\right).$$

Under squared-error loss, the Bayes estimator is the posterior mean. Define the **shrinkage factor**
$$\zeta = \frac{1}{1 + \tau^2} \in (0, 1).$$

Rearranging the posterior mean as a **convex combination** of the prior mean $\mu$ and the data $X_i$:
$$\boxed{\ \hat\theta_i(\mu, \zeta) = \zeta \mu + (1 - \zeta)\, X_i.\ }$$

Weight $\zeta$ on the prior mean, weight $1 - \zeta$ on the data.

Interpretation of the endpoints:

- $\tau^2 \to \infty$ (diffuse prior) $\Longrightarrow$ $\zeta \to 0$ $\Longrightarrow$ no shrinkage, $\hat\theta_i \to X_i$ (the MLE).
- $\tau^2 \to 0$ (concentrated prior) $\Longrightarrow$ $\zeta \to 1$ $\Longrightarrow$ full shrinkage, $\hat\theta_i \to \mu$.

### 3.3 Reading this as a linear function of $X_i$

It's useful to group the terms differently: the Bayes estimator is a **linear function of $X_i$**,
$$\hat\theta_i(\mu, \zeta) = \underbrace{\zeta \mu}_{\text{intercept}} + \underbrace{(1 - \zeta)}_{\text{slope}} \cdot X_i.$$

To apply this linear function, we need exactly two numbers: the slope $1 - \zeta$ and the intercept $\zeta \mu$. If someone gave us those two numbers, we could plug them in and beat the MLE. The catch: we don't know either $\mu$ or $\zeta$. But we have $d$ parallel observations that carry information about both.

---

## 4. Unknown $(\mu, \tau^2)$: Two Coefficients to Estimate

### 4.1 Two philosophies

- **Hierarchical Bayes.** Treat $(\mu, \tau^2)$ as more unknowns and place priors on them; compute their full posterior given the data.
- **Empirical Bayes.** Estimate $(\mu, \tau^2)$ directly from the data, and plug in.

Either way, the game is the same: we need to pick numerical values for the intercept $\zeta \mu$ and the slope $1 - \zeta$ in the linear-shrinkage formula from §3. Those two numbers are the only things the estimator depends on.

### 4.2 Hierarchical Bayes, in one step

The posterior mean of a linear function of $X_i$ is that same linear function applied with the posterior means of the coefficients. The key move is the tower property: condition first on the hyperparameters $(\mu, \zeta)$, take the inner expectation using §3, then average over the posterior on $(\mu, \zeta)$:

$$
\begin{aligned}
\hat\theta_i^{\text{HB}}
&= E[\theta_i \mid X] \\
&= E\!\left[\, E[\theta_i \mid \mu, \zeta, X]\ \big|\ X \,\right] && (\text{tower property}) \\
&= E\!\left[\,\zeta \mu + (1 - \zeta) X_i\ \big|\ X\right]         && (\S 3) \\
&= E[\zeta \mu \mid X] + \bigl(1 - E[\zeta \mid X]\bigr)\, X_i.
\end{aligned}
$$

$$\boxed{\ \hat\theta_i^{\text{HB}} = E[\zeta \mu \mid X] + \bigl(1 - E[\zeta \mid X]\bigr)\, X_i.\ }$$

**In words:** the hierarchical-Bayes estimator applies the *same linear function of $X_i$* as the oracle, except that the two unknowns $\zeta \mu$ and $\zeta$ are replaced by their **posterior means given the full data set**. Both quantities are estimated from all $d$ observations — and then fed into the same linear-shrinkage rule to estimate each individual $\theta_i$.

So it turns out hierarchical Bayes, despite sounding fancy, is doing something very concrete: it estimates the slope and intercept of the oracle's best shrinkage rule (using posterior means from the full data set), and plugs them in. It's a **plug-in estimator in disguise**.

### 4.3 Empirical Bayes, by analogy

Once we see hierarchical Bayes this way — as plugging data-driven estimates of the slope and intercept into the oracle's formula — a natural question is: *do we really need the full Bayesian machinery to get those two numbers?* The hyperparameters $\mu$ and $\zeta$ are identifiable from the marginal distribution of the $X_i$'s; maybe we can just estimate them some other way and plug them in.

That is **empirical Bayes**: use **point estimates** of the same two quantities, by any convenient method:
$$\hat\theta_i^{\text{EB}} = \hat\zeta \hat\mu + (1 - \hat\zeta)\, X_i.$$

We'll see in §5 that the sample mean and sample variance give us $\hat\mu$ and $\hat\zeta$ almost for free.

Three estimators — **oracle Bayes**, **hierarchical Bayes**, **empirical Bayes** — all apply a linear function of $X_i$. They differ only in what we plug in for the intercept and slope:

| Estimator | Intercept | Slope |
|---|---|---|
| Oracle Bayes (known $\mu$, $\zeta$) | $\zeta \mu$ | $1 - \zeta$ |
| Hierarchical Bayes | $E[\zeta \mu \mid X]$ | $1 - E[\zeta \mid X]$ |
| Empirical Bayes | $\hat\zeta\, \hat\mu$ | $1 - \hat\zeta$ |

**Key point:** each individual $\theta_i$ uses only its own $X_i$, but the *coefficients* are estimated from the entire data set. That is how information gets shared across the $d$ problems.

For large $d$ we'll see in §6 that the posteriors on $\mu$ and $\zeta$ both concentrate tightly, so hierarchical Bayes and empirical Bayes agree — and both approach the oracle.

---

## 5. Reading $(\mu, \tau^2)$ Off the Marginal Distribution

### 5.1 The marginal distribution of $X$

Integrating $\theta_i$ out of the hierarchical model: $X_i = \theta_i + \varepsilon_i$ with $\theta_i \sim N(\mu, \tau^2)$ and $\varepsilon_i \sim N(0, 1)$ independently. So
$$X_i \stackrel{iid}{\sim} N(\mu,\ 1 + \tau^2).$$

The marginal mean is $\mu$; the marginal variance is $1 + \tau^2$. So the **center of the $X_i$'s pins down $\mu$, and the spread pins down $\tau^2$**.

In [ ]:
# Generate the running-example dataset
d_run = 200
mu_true = 3.0
tau2_true = 4.0

rng_run = np.random.default_rng(0)
theta_run = rng_run.normal(mu_true, np.sqrt(tau2_true), size=d_run)
X_run = rng_run.normal(theta_run, 1.0)

print(f"d = {d_run}, true mu = {mu_true}, true tau^2 = {tau2_true}")

In [ ]:
# Figure 1: histogram of the observed X_i values
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(X_run, bins=20, density=True,
        color=COLOR_LIKELIHOOD, alpha=0.45, edgecolor='white')
ax.set_xlabel('$X_i$')
ax.set_ylabel('density')
ax.set_title(f'Histogram of the observed $X_i$ values (d = {d_run})')
plt.tight_layout()
plt.show()

*__Figure 1.__ Histogram of the $d = 200$ observed values $X_i$ from the running example. The center of the histogram estimates $\mu$; the spread estimates $1 + \tau^2$.*

### 5.2 Estimating the two things we need

Recall from §4 that empirical Bayes needs two numbers: $\hat\mu$ and $\hat\zeta$ (from which the intercept and slope follow). Both can be read off the **sample mean and sample variance** of the $X_i$'s — the summaries students have been computing since intro stats.

From the marginal:

- $E[X_i] = \mu$, so $\hat\mu = \bar X$.
- $\text{Var}(X_i) = 1 + \tau^2 = 1/\zeta$, so $\hat\zeta = 1/S^2$, where $S^2 = \frac{1}{d-1}\sum_i (X_i - \bar X)^2$.

The slope is $1 - \hat\zeta$ and the intercept is $\hat\zeta\,\hat\mu$.

(If $S^2 < 1$ — unusual for $d$ large under the model — we'd clip $\hat\zeta$ at $1$, i.e., full shrinkage to $\bar X$.)

**Moral:** the sample mean and sample variance already tell us everything we need.

In [ ]:
# Apply empirical Bayes to the running example
mu_hat = X_run.mean()                     # Xbar
S2_hat = X_run.var(ddof=1)                 # sample variance, d-1 denominator
zeta_hat = 1 / max(1.0, S2_hat)            # slope coefficient
zeta_mu_hat = zeta_hat * mu_hat            # intercept coefficient

# EB estimate = intercept + slope * X_i
theta_eb = zeta_mu_hat + (1 - zeta_hat) * X_run

# Comparisons
theta_mle = X_run
zeta_oracle = 1 / (1 + tau2_true)
theta_oracle = zeta_oracle * mu_true + (1 - zeta_oracle) * X_run

print(f"Xbar = {mu_hat:.3f}    (truth mu = {mu_true})")
print(f"S^2  = {S2_hat:.3f}    (target 1 + tau^2 = {1 + tau2_true})")
print()
print(f"slope      1 - zeta_hat     = {1 - zeta_hat:.3f}   (oracle {1 - zeta_oracle:.3f})")
print(f"intercept  zeta_hat * mu_hat = {zeta_mu_hat:.3f}   (oracle {zeta_oracle * mu_true:.3f})")
print()
print(f"MLE    MSE: {np.mean((theta_mle   - theta_run)**2):.3f}")
print(f"EB     MSE: {np.mean((theta_eb    - theta_run)**2):.3f}")
print(f"Oracle MSE: {np.mean((theta_oracle - theta_run)**2):.3f}")

In [ ]:
# Figure 2: MLE vs EB on the running example
fig, ax = plt.subplots(figsize=(8, 5.5))

diag = np.array([theta_run.min() - 1, theta_run.max() + 1])
ax.plot(diag, diag, 'k:', alpha=0.4, label='identity (perfect estimate)')

for t, mle, eb in zip(theta_run, theta_mle, theta_eb):
    ax.plot([t, t], [mle, eb], color='gray', alpha=0.15, linewidth=0.5)

ax.scatter(theta_run, theta_mle, color=COLOR_MLE, alpha=0.55, s=30,
           label='MLE:  $\\hat\\theta_i = X_i$')
ax.scatter(theta_run, theta_eb, color=COLOR_EB, alpha=0.55, s=30,
           label=fr'EB:  $\hat\zeta \hat\mu + (1 - \hat\zeta) X_i$,  $\hat\zeta = {zeta_hat:.2f}$')

ax.set_xlabel(r'true $\theta_i$')
ax.set_ylabel('estimate')
ax.legend(loc='upper left')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
plt.show()

*__Figure 2.__ Estimates against the true $\theta_i$'s (we can compare because this is simulated). Blue dots are the MLE $\hat\theta_i = X_i$; purple dots are the empirical-Bayes estimates $\hat\zeta\hat\mu + (1 - \hat\zeta) X_i$. Gray segments connect the two estimates for each $i$. Every EB estimate is produced by the *same* linear function of $X_i$ — one line with slope $1 - \hat\zeta \approx 0.78$ and intercept $\hat\zeta\hat\mu \approx 0.65$ — where the coefficients were estimated from the whole data set. The EB estimates sit collectively closer to the identity line than the MLEs.*

---

## 6. Why It Works: the Hyperparameter Concentrates

We now come to the central point of the lecture. With a single problem, there is only so much the data can tell us about $\tau^2$. But with $d$ parallel problems, the marginal distribution $N(0, 1 + \tau^2)$ gets $d$ independent draws from it — and that is an enormous amount of information about the population. We'll make this concrete in two settings.

### 6.1 Concentration in the Gaussian-Gaussian model

We'll use an improper **flat prior** on $(\mu, \tau^2)$ — the simplest non-informative choice — which is proper once we have $d \geq 4$ observations. After integrating out the latent $\theta_i$'s we're doing standard Bayesian inference for an iid sample $X_i \sim N(\mu, 1+\tau^2)$, and the marginal posteriors are the familiar ones from any Bayesian textbook:

$$\mu \mid X \ \sim\ \bar X + S\sqrt{\tfrac{d-1}{d(d-3)}}\, t_{d-3}, \qquad (1 + \tau^2) \mid X \ \sim\ \text{InverseGamma}\!\left(\tfrac{d-3}{2},\ \tfrac{(d-1) S^2}{2}\right).$$

The posterior on $\mu$ is a $t$-distribution centered at the sample mean. The posterior on $\tau^2 = \sigma^2 - 1$ is a shifted inverse-gamma. Both concentrate as $d$ grows — we visualize this below.

In [ ]:
# Figure 3: marginal posteriors on mu (top row) and tau^2 (bottom row) for d = 5, 20, 100.
# Flat prior on (mu, tau^2); the model requires tau^2 > 0 (equivalently sigma^2 = 1+tau^2 > 1),
# so we evaluate the joint posterior on a grid restricted to sigma^2 > 1 and marginalize numerically.
d_values = [10, 50, 200]

fig, axes = plt.subplots(2, len(d_values), figsize=(14, 7))

mu_grid = np.linspace(0, 6, 400)
tau2_grid = np.linspace(0, 20, 400)     # tau^2 >= 0
sigma2_grid = 1 + tau2_grid              # sigma^2 >= 1 (truncation built in)

d_mu  = mu_grid[1]  - mu_grid[0]
d_t2  = tau2_grid[1] - tau2_grid[0]

for col, (d_demo, seed) in enumerate(zip(d_values, [27, 6, 0])):
    rng_demo = np.random.default_rng(seed)
    theta_demo = rng_demo.normal(mu_true, np.sqrt(tau2_true), size=d_demo)
    X_demo = rng_demo.normal(theta_demo, 1.0)
    Xbar = X_demo.mean()
    ss = np.sum((X_demo - Xbar)**2)   # (d-1) * S^2

    # Joint log-posterior under flat prior, as a function of (mu, sigma^2) with sigma^2 >= 1
    MU, SIG2 = np.meshgrid(mu_grid, sigma2_grid, indexing='ij')
    log_joint = (-d_demo / 2) * np.log(SIG2) - ss / (2 * SIG2) - d_demo * (MU - Xbar)**2 / (2 * SIG2)
    log_joint -= log_joint.max()
    joint = np.exp(log_joint)
    joint /= joint.sum() * d_mu * d_t2     # normalize over (mu, tau^2) grid

    mu_post   = joint.sum(axis=1) * d_t2
    tau2_post = joint.sum(axis=0) * d_mu

    # Top row: posterior on mu
    ax_mu = axes[0, col]
    ax_mu.plot(mu_grid, mu_post, color=COLOR_POSTERIOR, linewidth=2)
    ax_mu.fill_between(mu_grid, 0, mu_post, color=COLOR_POSTERIOR, alpha=0.2)
    ax_mu.axvline(mu_true, color='k', linestyle='--', label=fr'true $\mu = {mu_true:g}$')
    ax_mu.set_title(fr'$d = {d_demo}$')
    ax_mu.set_xlabel(r'$\mu$')
    ax_mu.set_xlim(0, 6)

    # Bottom row: posterior on tau^2
    ax_t = axes[1, col]
    ax_t.plot(tau2_grid, tau2_post, color=COLOR_POSTERIOR, linewidth=2)
    ax_t.fill_between(tau2_grid, 0, tau2_post, color=COLOR_POSTERIOR, alpha=0.2)
    ax_t.axvline(tau2_true, color='k', linestyle='--', label=fr'true $\tau^2 = {tau2_true:g}$')
    ax_t.set_xlabel(r'$\tau^2$')
    ax_t.set_xlim(0, 20)

axes[0, 0].set_ylabel(r'posterior density of $\mu$')
axes[1, 0].set_ylabel(r'posterior density of $\tau^2$')
axes[0, 0].legend(loc='upper right')
axes[1, 0].legend(loc='upper right')
plt.tight_layout()
plt.show()

*__Figure 3.__ Marginal posterior densities of the hyperparameters $\mu$ (top row) and $\tau^2$ (bottom row), for three sample sizes $d \in \{10, 50, 200\}$. True values ($\mu = 3$, $\tau^2 = 4$) marked by dashed lines. At $d = 10$ both posteriors are broad — we're quite uncertain about the population. By $d = 200$, both are sharply concentrated around the truth. **With many parallel problems, we effectively know both hyperparameters.***

### 6.2 Risk comparison: empirical Bayes approaches the oracle

If the hyperparameter is effectively known, then empirical Bayes should deliver risk close to the oracle Bayes estimator (which truly knows $\tau^2$). We verify this by simulation.

For hierarchical Bayes we use the same flat prior as in §6.1: $E[\zeta \mid X]$ is computed as the truncated-inverse-gamma posterior mean (a small adjustment of the untruncated $(d-3)/((d-1)S^2)$), and $E[\zeta\mu \mid X] = \bar X \cdot E[\zeta \mid X]$.

In [ ]:
# Figure 4: average MSE vs d for MLE, EB, HB, Oracle
from scipy.stats import gamma as _gamma
d_range = np.array([5, 10, 20, 50, 100, 200])
n_reps = 10000

def hb_zeta_mean(d, ss):
    """E[zeta | X] under flat prior on (mu, tau^2), truncated to sigma^2 > 1.

    1/sigma^2 ~ Gamma((d-3)/2, rate=ss/2) truncated to (0, 1), so
    E[zeta | X] = (alpha/beta) * F_{Gamma(alpha+1, beta)}(1) / F_{Gamma(alpha, beta)}(1).
    """
    if d < 4:
        return np.nan
    alpha, beta = (d - 3) / 2, ss / 2
    denom = _gamma.cdf(1, a=alpha, scale=1 / beta)
    if denom <= 0:
        return np.nan
    return (alpha / beta) * _gamma.cdf(1, a=alpha + 1, scale=1 / beta) / denom

mse_mle = np.zeros(len(d_range))
mse_eb = np.zeros(len(d_range))
mse_hb = np.zeros(len(d_range))
mse_oracle = np.zeros(len(d_range))

zeta_oracle = 1 / (1 + tau2_true)
rng_mse = np.random.default_rng(42)

for i, d in enumerate(d_range):
    mle_sse = np.zeros(n_reps)
    eb_sse  = np.zeros(n_reps)
    hb_sse  = np.zeros(n_reps)
    or_sse  = np.zeros(n_reps)
    for r in range(n_reps):
        theta = rng_mse.normal(mu_true, np.sqrt(tau2_true), size=d)
        X = rng_mse.normal(theta, 1.0)
        Xbar = X.mean()
        ss = np.sum((X - Xbar)**2)
        S2 = ss / (d - 1)

        # MLE
        mle_sse[r] = np.mean((X - theta)**2)

        # Empirical Bayes: Xbar and S^2 moment plug-in
        zeta_eb = 1 / max(1.0, S2)
        eb = Xbar + (1 - zeta_eb) * (X - Xbar)
        eb_sse[r] = np.mean((eb - theta)**2)

        # Hierarchical Bayes: flat prior, posterior-mean coefficients
        zeta_hb = hb_zeta_mean(d, ss)
        hb = Xbar + (1 - zeta_hb) * (X - Xbar)
        hb_sse[r] = np.mean((hb - theta)**2)

        # Oracle
        oracle = mu_true + (1 - zeta_oracle) * (X - mu_true)
        or_sse[r] = np.mean((oracle - theta)**2)

    mse_mle[i]    = mle_sse.mean()
    mse_eb[i]     = eb_sse.mean()
    mse_hb[i]     = hb_sse.mean()
    mse_oracle[i] = or_sse.mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(d_range, mse_mle,    'o-',  color=COLOR_MLE,    label='MLE   $\\hat\\theta_i = X_i$')
ax.plot(d_range, mse_eb,     'o-',  color=COLOR_EB,     label='Empirical Bayes')
ax.plot(d_range, mse_hb,     's-',  color=COLOR_HB,     label='Hierarchical Bayes (flat prior)')
ax.plot(d_range, mse_oracle, 'o--', color=COLOR_ORACLE, label=fr'Oracle (knows $\mu, \tau^2$)')
ax.set_xscale('log')
ax.set_xlabel('number of parallel problems $d$')
ax.set_ylabel('average per-coordinate MSE')
ax.legend()
plt.tight_layout()
plt.show()

*__Figure 4.__ Per-coordinate MSE, averaged over 10,000 simulated datasets, as a function of the number of parallel problems $d$. The MLE (blue) has risk $1$ for every $d$ — it gains nothing from more problems. The oracle Bayes estimator (black dashed), which uses the true $(\mu, \tau^2)$, has risk $\tau^2/(1+\tau^2) = 0.8$. Empirical Bayes (purple) and hierarchical Bayes (magenta) both shrink toward the sample mean using a data-driven shrinkage factor; they start above the oracle for small $d$ (hyperparameters poorly estimated) but **converge to the oracle as $d$ grows**. That is the payoff of the concentration in Figure 3. HB and EB are nearly indistinguishable — they use very similar shrinkage factors, differing only in the finite-sample correction.*

### 6.3 Gamma-Poisson: concentration via Gibbs trace plots

The concentration story isn't specific to the Gaussian-Gaussian model. It applies to any hierarchical Bayes model with enough parallel problems. We illustrate this with the **real Premier League goal-scoring data** from Worksheet 4 — the first 10 matches of the 2023-24 season for all 20 teams.

**Model.** For teams $i = 1, \ldots, d$ and matches $j = 1, \ldots, m$:
$$\theta_i \mid \beta \stackrel{iid}{\sim} \text{Gamma}(\alpha, \beta), \qquad X_{ij} \mid \theta_i \sim \text{Poisson}(\theta_i).$$

As in Worksheet 4, we hold the shape $\alpha$ fixed (at $\alpha = 15$) for simplicity and put a diffuse $\text{Gamma}(a, b)$ hyperprior on $\beta$. A fully principled analysis would sample $\alpha$ too, but with only $d = 20$ teams the data doesn't pin down $\alpha$ and $\beta$ individually very tightly — fixing $\alpha$ lets us focus on the concentration story for $\beta$.

**Gibbs conditional for team rates** (conjugacy from Worksheet 4): with $G_i = \sum_j X_{ij}$,
$$\theta_i \mid \beta, X \sim \text{Gamma}(\alpha + G_i,\ \beta + m).$$

**Gibbs conditional for the hyperparameter** (Gamma-Gamma conjugacy):
$$\beta \mid \theta \sim \text{Gamma}\!\left(a + d\alpha,\ b + \sum_i \theta_i\right).$$

Both conditionals are closed-form — this is a fully conjugate Gibbs sampler.

In [ ]:
# Real Premier League data: goals per match, 20 teams, first 10 matches of 2023-24
# Source: football-data.co.uk (same dataset as Worksheet 4)
teams = ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton',
         'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham',
         'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle',
         "Nott'm Forest", 'Sheffield United', 'Tottenham', 'West Ham', 'Wolves']

goals = np.array([
    [2, 2, 3, 3, 6, 3, 0, 4, 3, 1],  # Arsenal
    [3, 5, 1, 0, 2, 0, 0, 3, 1, 1],  # Aston Villa
    [1, 1, 2, 2, 2, 1, 0, 1, 1, 2],  # Bournemouth
    [1, 2, 2, 3, 0, 0, 3, 1, 3, 1],  # Brentford
    [0, 0, 3, 4, 0, 2, 1, 1, 0, 2],  # Brighton
    [1, 2, 5, 2, 0, 2, 1, 0, 0, 0],  # Burnley
    [2, 2, 0, 2, 3, 2, 4, 5, 4, 1],  # Chelsea
    [1, 0, 1, 3, 1, 2, 2, 4, 0, 0],  # Crystal Palace
    [1, 1, 2, 1, 2, 1, 1, 1, 3, 3],  # Everton
    [3, 1, 1, 0, 2, 3, 0, 0, 0, 5],  # Fulham
    [4, 1, 3, 4, 1, 3, 4, 1, 2, 2],  # Liverpool
    [1, 2, 1, 4, 0, 1, 1, 1, 3, 2],  # Luton
    [5, 3, 4, 3, 5, 6, 3, 4, 0, 0],  # Man City
    [4, 0, 1, 1, 3, 3, 1, 0, 2, 2],  # Man United
    [2, 3, 1, 1, 1, 4, 4, 1, 1, 0],  # Newcastle
    [1, 0, 3, 1, 0, 1, 3, 2, 0, 1],  # Nott'm Forest
    [2, 0, 0, 0, 1, 1, 2, 0, 2, 1],  # Sheffield United
    [3, 1, 0, 5, 2, 3, 2, 1, 3, 1],  # Tottenham
    [1, 2, 0, 3, 1, 1, 0, 2, 0, 2],  # West Ham
    [3, 0, 1, 1, 2, 4, 1, 1, 1, 1],  # Wolves
])
d_teams, m_matches = goals.shape
G_team = goals.sum(axis=1)            # total goals per team over 10 matches

print(f"{d_teams} teams, {m_matches} matches each")
print(f"mean G_i = {G_team.mean():.2f}, var G_i = {G_team.var(ddof=1):.2f}")

**Sanity check on the model.** Before running the Gibbs sampler, let's verify that the Gamma-Poisson hierarchy is a sensible description of this data. If $\theta_i \sim \text{Gamma}(\alpha, \beta)$ and $G_i = \sum_j X_{ij} \mid \theta_i \sim \text{Poisson}(m\theta_i)$, then marginally $G_i \sim \text{NegBin}(\alpha, \beta/(\beta + m))$.

Holding $\alpha = 15$ (as in the Gibbs sampler below) and computing the MLE of $\beta$, we can compare the negative-binomial fit against the empirical distribution of team totals.

In [ ]:
# Figure 5: Histogram of G_i + NegBin(alpha=15, beta_MLE) overlay
alpha_fit = 15.0
beta_mle = alpha_fit * d_teams * m_matches / G_team.sum()
p_mle = beta_mle / (beta_mle + m_matches)

k_grid = np.arange(0, G_team.max() + 6)
nb_pmf = stats.nbinom.pmf(k_grid, n=alpha_fit, p=p_mle)

fig, ax = plt.subplots(figsize=(10, 4))
bin_edges = np.arange(0, G_team.max() + 6, 5)   # width-5 bins
ax.hist(G_team, bins=bin_edges, density=True, color=COLOR_LIKELIHOOD,
        alpha=0.4, edgecolor='white', label='observed $G_i$ (20 teams)')
ax.plot(k_grid, nb_pmf, 'o-', color=COLOR_POSTERIOR, linewidth=1.5, markersize=5,
        label=fr'NegBin MLE fit  ($\alpha = {alpha_fit:g}$, $\hat\beta = {beta_mle:.2f}$)')
ax.set_xlabel('total goals over 10 matches, $G_i$')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.show()

*__Figure 5.__ Histogram of total goals $G_i$ across the 20 Premier-League teams (first 10 matches, 2023-24), overlaid with the MLE negative-binomial fit under the Gamma-Poisson hierarchy (holding $\alpha = 15$). The empirical distribution and the fitted PMF line up reasonably well — the hierarchical Gamma-Poisson model is a credible description of the data.*

In [ ]:
# Gibbs sampler for the Gamma-Poisson hierarchy (alpha fixed at 15)
alpha = 15.0                            # held fixed (as in WS04)
a_hyper, b_hyper = 1.0, 0.1             # beta ~ Gamma(1, 0.1) hyperprior

rng_ep = np.random.default_rng(101)

# Gibbs sampler (both updates closed-form)
n_iter = 2000
theta_trace = np.zeros((n_iter, d_teams))
beta_trace = np.zeros(n_iter)

beta_cur = 5.0
for t in range(n_iter):
    theta_cur = rng_ep.gamma(alpha + G_team, 1 / (beta_cur + m_matches))
    theta_trace[t] = theta_cur

    beta_cur = rng_ep.gamma(a_hyper + d_teams * alpha,
                            1 / (b_hyper + theta_cur.sum()))
    beta_trace[t] = beta_cur

burnin = 200
print(f"alpha held at {alpha:g}")
print(f"posterior mean of beta:  {beta_trace[burnin:].mean():.3f}")
print(f"posterior SD   of beta:  {beta_trace[burnin:].std():.3f}")

In [ ]:
# Figure 6: Gibbs trace plots - beta (tight) vs theta_i (wide)
iters_shown = np.arange(burnin, n_iter)

# Three teams of contrasting form, per WS04: Sheffield United / Man United / Man City
team_picks = [teams.index('Sheffield United'),
              teams.index('Man United'),
              teams.index('Man City')]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: beta (hyperparameter)
axes[0].plot(iters_shown, beta_trace[burnin:], color=COLOR_POSTERIOR, linewidth=0.8)
axes[0].set_title(r'Hyperparameter $\beta$: concentrated')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel(r'$\beta$')
axes[0].set_ylim(0, 25)

# Right: three theta_i traces (three distinct colors) with MLE reference lines
for idx, c in zip(team_picks, TEAM_COLORS):
    mle_rate = G_team[idx] / m_matches
    axes[1].plot(iters_shown, theta_trace[burnin:, idx],
                 color=c, linewidth=0.8, alpha=0.85,
                 label=f'{teams[idx]}  (MLE rate {mle_rate:.2f})')
    axes[1].axhline(mle_rate, color=c, linestyle=':', linewidth=0.8, alpha=0.6)
axes[1].set_title(r'Individual rates $\theta_i$: much more uncertain')
axes[1].set_xlabel('iteration')
axes[1].set_ylabel(r'$\theta_i$ (goals per match)')
axes[1].legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

*__Figure 6.__ Gibbs-sampler trace plots on the 2023-24 Premier League data, after burn-in. **Left:** trace of the hyperparameter $\beta$ — a tight band indicating a narrow posterior (we don't know the "truth" for $\beta$ since this is real data, but the trace mixes well in a narrow range). **Right:** traces of three individual team rates $\theta_i$ for Sheffield United, Man United, and Man City — in three distinct colors, with each team's raw MLE rate $G_i/m$ shown as a dotted reference line. These traces wander over a much wider range than $\beta$, reflecting substantial posterior uncertainty about any single team's rate. The moral: **we don't know each team precisely, but we know the population precisely — and that's exactly what lets us shrink each team's estimate toward a reliable target.***

### 6.4 Unifying idea

Whether the hyperparameter is the Gaussian prior variance $\tau^2$, the Beta parameters $(\alpha, \beta)$ in Lecture 8 baseball, or the Gamma parameter $\beta$ in the Premier League model, the mechanism is the same:

- **Many parallel problems** ⟹ the marginal distribution of the data gives a lot of information about the population.
- ⟹ the **posterior on the hyperparameter concentrates**.
- ⟹ the "right shrinkage" for each individual problem is effectively known.
- ⟹ we can shrink each parameter confidently toward a reliable, data-learned population target.

Individual parameters remain uncertain, but the *relationships among them* — as captured by the hyperparameter — become sharp. That is why hierarchical and empirical Bayes are such powerful tools for problems with many related units.

---

## 7. Summary

### Key formulas

Every Bayes-style estimator in this lecture applies the same **linear function of $X_i$**: $\hat\theta_i = \text{intercept} + \text{slope} \cdot X_i$. The three versions differ only in what plays the role of intercept and slope.

| Estimator | Intercept | Slope |
|---|---|---|
| MLE | $0$ | $1$ |
| Oracle Bayes (known $\mu, \zeta$) | $\zeta \mu$ | $1 - \zeta$ |
| Hierarchical Bayes | $E[\zeta\mu \mid X]$ | $1 - E[\zeta \mid X]$ |
| Empirical Bayes | $\hat\zeta\,\hat\mu = \bar X / S^2$ | $1 - \hat\zeta = 1 - 1/S^2$ |

### Key takeaways

1. **The Bayes estimator is a linear function of $X_i$.** With $(\mu, \zeta)$ known, $\hat\theta_i = \zeta\mu + (1 - \zeta) X_i$. Each individual estimate uses only its own $X_i$, but through a linear rule shared by all $d$ problems.

2. **Hierarchical Bayes and empirical Bayes just estimate the two coefficients.** Both plug estimates of $\mu$ and $\zeta$ — derived from the *full* data set — into the same linear function. That's how information is shared.

3. **Empirical Bayes uses familiar summaries.** The sample mean $\bar X$ and sample variance $S^2$ already encode what we need: $\hat\mu = \bar X$, $\hat\zeta = 1/S^2$. More generally — for hierarchical models that aren't Gaussian — empirical Bayes picks the hyperparameters by **maximizing the marginal likelihood** of the data (marginalizing out the individual parameters, then taking the MLE of the hyperparameters). For example, in the Premier League Gamma-Poisson model, we could estimate $(\alpha, \beta)$ by maximizing the integrated likelihood over the $\theta_i$'s, then plug those estimates into the Gamma-Poisson Bayes update for each team. The sample mean and sample variance turn out to be the marginal-MLE shortcut in the special Gaussian case.

4. **The engine: hyperparameter concentration.** With many parallel problems, the posterior on the hyperparameters concentrates (Figure 3) — even while the posteriors on individual parameters do not (Figure 6). That's why hierarchical and empirical Bayes effectively use the optimal shrinkage without knowing the truth.

5. **Connections.** This is the Gaussian cousin of the hierarchical Bayes we saw for baseball (Lecture 8, Beta-Binomial) and Premier League scoring (Worksheet 4, Gamma-Poisson). In all three settings, the same mechanism — well-identified hyperparameters pinned down by many parallel observations — is what makes "borrowing strength" work.